# 04 — SQL Analytics on Gold

Register silver and gold tables as temp views, then run SQL queries that exercise:

- aggregations + group by
- joins across verticals
- window functions (`SUM() OVER`, `RANK() OVER`)
- date functions and time-of-day patterns
- CTEs (`WITH ...`)

Every query is plain Spark SQL — what an analyst would write against the lakehouse.

## Setup — register all silver and gold tables as temp views

In [ ]:
import sys
from pathlib import Path
if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

from conf.spark import get_spark, SILVER_DIR, GOLD_DIR
spark = get_spark("SqlAnalytics")

for d in SILVER_DIR.iterdir():
    if d.is_dir():
        spark.read.format("delta").load(str(d)).createOrReplaceTempView(d.name)
for d in GOLD_DIR.iterdir():
    if d.is_dir():
        spark.read.format("delta").load(str(d)).createOrReplaceTempView(d.name)

spark.sql("SHOW TABLES").show(truncate=False)

## Q1 — Top 10 merchants by flagged-rate

Which merchants see the highest share of flagged (suspected-fraud) transactions? Filter to merchants with ≥10 transactions so a single flagged txn doesn't dominate the ranking.

In [ ]:
spark.sql("""
    SELECT merchant_name,
           merchant_category,
           COUNT(*)                                            AS txn_count,
           SUM(CAST(is_flagged AS INT))                        AS flagged_count,
           ROUND(100.0 * AVG(CAST(is_flagged AS INT)), 2)      AS flagged_pct
    FROM   card_transactions
    GROUP  BY merchant_name, merchant_category
    HAVING COUNT(*) >= 10
    ORDER  BY flagged_pct DESC, txn_count DESC
    LIMIT  10
""").show(truncate=False)

## Q2 — Monthly revenue per vertical with rolling 3-month average

Window function over a vertical's monthly totals. Useful for trend dashboards.

In [ ]:
spark.sql("""
    WITH monthly AS (
        SELECT vertical,
               DATE_TRUNC('month', txn_date) AS month,
               SUM(amount_total)             AS revenue
        FROM   daily_transactions
        GROUP  BY vertical, DATE_TRUNC('month', txn_date)
    )
    SELECT vertical,
           month,
           revenue,
           ROUND(
             AVG(revenue) OVER (
               PARTITION BY vertical
               ORDER BY month
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
             ), 2) AS rolling_3m_avg
    FROM monthly
    ORDER BY vertical, month
""").show(truncate=False)

## Q3 — Customers active in all four verticals (cross-vertical)

The bank's prime cross-sell target — customers with at least one card txn, one loan, one account, and one payment.

In [ ]:
spark.sql("""
    SELECT c.customer_id,
           c.full_name,
           c.city,
           c.card_txn_count,
           c.loan_count,
           c.account_count,
           c.payment_count,
           ROUND(c.card_spend_total + c.payment_total, 2) AS lifetime_value
    FROM   customer_360 c
    WHERE  c.card_txn_count > 0
      AND  c.loan_count     > 0
      AND  c.account_count  > 0
      AND  c.payment_count  > 0
    ORDER  BY lifetime_value DESC
    LIMIT  10
""").show(truncate=False)

## Q4 — Loan default rate by loan_type and city

Joins `loan_portfolio` to `customers` to slice defaults by geography. The kind of view risk teams want first.

In [ ]:
spark.sql("""
    SELECT c.city,
           p.loan_type,
           COUNT(*)                                       AS loans,
           SUM(CAST(p.is_default AS INT))                 AS defaults,
           ROUND(100.0 * AVG(CAST(p.is_default AS INT)), 2) AS default_pct
    FROM   loan_portfolio p
    JOIN   customers      c ON c.customer_id = p.customer_id
    GROUP  BY c.city, p.loan_type
    HAVING COUNT(*) >= 2
    ORDER  BY default_pct DESC, loans DESC
""").show(50, truncate=False)

## Q5 — Reconstruct a chronological running balance per account

Each `account_transactions` row carries a `balance_after` snapshot, but those snapshots were assigned in the order the generator emitted rows — not strictly chronological. Here we use a window function to compute the **chronological** running balance: cumulative sum of signed amounts, ordered by `txn_at` per account. This is the canonical pattern for late-arriving event reconstruction.

In [ ]:
spark.sql("""
    WITH signed AS (
        SELECT account_id,
               txn_id,
               txn_at,
               txn_type,
               amount,
               CASE WHEN txn_type IN ('withdrawal','fee') THEN -amount ELSE amount END AS signed_amount
        FROM   account_transactions
    )
    SELECT account_id,
           txn_id,
           txn_at,
           txn_type,
           amount,
           SUM(signed_amount) OVER (
             PARTITION BY account_id ORDER BY txn_at
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS running_net_change
    FROM   signed
    WHERE  account_id = 'A0001'
    ORDER  BY txn_at
    LIMIT  10
""").show(truncate=False)

## Q6 — Hour-of-day fraud distribution

Fraudulent (flagged) transactions cluster at odd hours. Plot-ready output for the security team.

In [ ]:
spark.sql("""
    SELECT HOUR(transaction_at) AS hour_of_day,
           COUNT(*)                                       AS total_txns,
           SUM(CAST(is_flagged AS INT))                   AS flagged_txns,
           ROUND(100.0 * AVG(CAST(is_flagged AS INT)), 2) AS flagged_pct
    FROM   card_transactions
    GROUP  BY HOUR(transaction_at)
    ORDER  BY hour_of_day
""").show(24, truncate=False)

## Q7 — Top-3 spending customers per city (RANK window)

Classic `RANK() OVER (PARTITION BY ...)` pattern.

In [ ]:
spark.sql("""
    WITH ranked AS (
        SELECT c.city,
               c.full_name,
               ROUND(c.card_spend_total, 2) AS card_spend,
               RANK() OVER (PARTITION BY c.city ORDER BY c.card_spend_total DESC) AS rnk
        FROM   customer_360 c
        WHERE  c.card_spend_total > 0
    )
    SELECT city, full_name, card_spend, rnk
    FROM   ranked
    WHERE  rnk <= 3
    ORDER  BY city, rnk
""").show(50, truncate=False)

## Q8 — Payment success rate by mode

Operations dashboard: which payment rail is most reliable?

In [ ]:
spark.sql("""
    SELECT payment_mode,
           COUNT(*)                                                 AS attempts,
           SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END)      AS successes,
           ROUND(100.0 * AVG(CASE WHEN status = 'success' THEN 1.0 ELSE 0.0 END), 2) AS success_pct,
           ROUND(SUM(CASE WHEN status = 'success' THEN amount ELSE 0 END), 2) AS success_value
    FROM   payments
    GROUP  BY payment_mode
    ORDER  BY success_pct DESC
""").show(truncate=False)

## Summary

Eight queries covering aggregations, multi-vertical joins, rolling windows, ranks, time-of-day patterns, and chronological running totals. The remaining notebooks (05, 06) cover streaming.